# 08a THINGS-EEG1, converted to the THINGS-EEG2 layout

**Kaggle:** CPU, Internet on (OpenNeuro's public bucket). **Inputs:** none, or this notebook's own earlier output to
resume.

THINGS-EEG1 (Grootswagers et al. 2022, OpenNeuro ds003825) has 50 people who saw THINGS images in rapid streams at
10 Hz: 22,248 training images once each, then 200 validation images 12 times each (people 1 and 6 have no
validation part). Every person is converted to THINGS-EEG2's release format so the same pipeline runs on them: the
authors' continuous files (0.1 to 100 Hz, average reference, 250 Hz), the 17 occipital and parietal channels
THINGS-EEG2 keeps, epochs from -200 to 800 ms, baseline on the 200 ms before the image, 100 Hz, and multivariate
noise normalization. It also writes the image lists, the k-image subsets and an audit with each person's noise
ceiling on the validation images.

In [ ]:
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
import json
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

try:
    import mne
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne"], check=True)
    import mne
from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.data.splits import nested_subsets
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset
from cross_subject_encoding.utils.preflight import preflight
from cross_subject_encoding.utils.selfcheck import check, headline

SUBJECTS = e1.SUBJECTS
OUT = C.DATA_PROCESSED / "eeg1"
WORK = Path("/kaggle/tmp/eeg1") if IS_KAGGLE else C.PROJECT_ROOT / "data" / "raw" / "things_eeg1"
TIME_BUDGET_H = 10.0
T0 = time.time()
minutes = lambda: f"{(time.time() - T0) / 60:.1f} min"
info = preflight("cross-subject-brain-encoding-08a-outputs", inputs=[], need_gpu=False, expect_output_gb=8.0)  # outputs plus the export's staging copy
print("mne", mne.__version__, "|", len(SUBJECTS), "people")

## The shared image lists

Every person saw the same 22,248 training images in their own random
order. The events tables say which image came when; the shared order is
taken from one person and checked against everyone.

In [ ]:
events = {}
for s in SUBJECTS:
    p = WORK / s / f"{s}_task-rsvp_events.tsv"
    if not p.exists():
        e1.fetch(e1.S3 + f"{s}/eeg/{s}_task-rsvp_events.tsv", p)
    events[s] = e1.read_events(p)
train_keys = e1.training_order(events["sub-02"])
val_keys = e1.validation_images(events["sub-02"])
same = {s: sorted(e1.image_key(ev[ev.isteststim != 1])) == sorted(train_keys) for s, ev in events.items()}
same_val = {s: e1.validation_images(ev) == val_keys for s, ev in events.items() if (ev.isteststim == 1).any()}
print(f"{len(train_keys)} training images, {len(val_keys)} validation images | training sets equal for "
      f"{sum(same.values())} of {len(same)} people, validation sets for {sum(same_val.values())} of {len(same_val)} | {minutes()}")
concept = lambda k: k.split("/")[0]
np.save(C.DATA_PROCESSED / "eeg1_image_metadata.npy", {"train_img_files": np.array([k.split("/")[1] for k in train_keys]),
        "train_img_concepts": np.array([concept(k) for k in train_keys]), "test_img_files": np.array([k.split("/")[1] for k in val_keys]),
        "test_img_concepts": np.array([concept(k) for k in val_keys])}, allow_pickle=True)
(C.DATA_PROCESSED / "eeg1_k_subsets.json").write_text(json.dumps(nested_subsets(len(train_keys), C.K_GRID, e1.HELD_OUT, C.SEEDS)))

## Every person

Download, convert, save, delete the download. People already in an
attached earlier output are loaded instead. The audit row records the
event counts and the noise ceiling: the paper's split-half correlation of
the validation responses (6 against 6 repetitions here, 40 against 40 in
THINGS-EEG2), averaged over channels, at its peak.

In [ ]:
rows, per_time = [], {}
done_before = Path("/kaggle/input") if IS_KAGGLE else None
for s in SUBJECTS:
    if (time.time() - T0) / 3600 > TIME_BUDGET_H:
        print(f"time budget reached before {s}")
        break
    t = time.time()
    if not same[s]:  # sub-06's shorter session lacks some training images; it has no validation part either
        rows.append({"person": s, "skipped": "training images differ from the shared list", "has_validation": False, "seconds": 0.0})
        print(f"{s}: skipped, its training images differ from the shared list ({(events[s].isteststim != 1).sum()} presentations)")
        continue
    prev = e1.person_file(done_before, s, "training") if done_before else None
    if prev is not None:
        for part in ("training", "test"):
            f = e1.person_file(done_before, s, part)
            if f is not None:
                OUT.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, OUT / f.name)
        rep = {"loaded from an earlier run": True}
    else:
        files = e1.download_subject(s, WORK)
        data, ch, sf, onsets = e1.read_continuous(files["set"])
        person = e1.build_person(data, ch, sf, onsets, events[s], train_keys)
        e1.save_release(OUT, s, person)
        rep = person["report"]
        del data, person
        shutil.rmtree(WORK / s, ignore_errors=True)
    test_file = e1.person_file(OUT, s, "test")
    nc_peak = nc_ms = np.nan
    if test_file is not None:
        test = np.load(test_file, allow_pickle=True).item()
        bio = np.asarray(test["preprocessed_eeg_data"], dtype=np.float32)
        nc = paper_correlation({}, bio)["noise_ceiling_low"].mean(0)
        per_time[s] = nc
        nc_peak, nc_ms = float(nc.max()), int(round(test["times"][int(nc.argmax())] * 1000))
    rows.append({"person": s, **rep, "has_validation": test_file is not None, "noise_ceiling_low_peak": nc_peak, "peak_ms": nc_ms,
                 "seconds": time.time() - t})
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "08a_audit.csv", index=False)
    print(f"{s}: {rep} | ceiling {nc_peak:.3f} at {nc_ms} ms | {time.time() - t:.0f} s | {minutes()}")
audit = pd.DataFrame(rows)
times_ms = np.round(np.arange(len(next(iter(per_time.values())))) * 10 - 200).astype(int)
np.savez_compressed(C.METRICS_DIR / "08a_noise_ceiling.npz", subjects=np.array(list(per_time)), times_ms=times_ms,
                    noise_ceiling_low=np.stack(list(per_time.values())))
audit.describe()

In [ ]:
import matplotlib.pyplot as plt

nc = np.stack(list(per_time.values()))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(times_ms, nc.T, color="0.7", lw=0.6)
ax[0].plot(times_ms, nc.mean(0), color="C3", lw=2, label="mean over people")
ax[0].set(xlabel="time from image onset (ms)", ylabel="split-half r (6 vs 6 repetitions)", title="Noise ceiling, validation images")
ax[0].legend()
ax[1].hist(audit.noise_ceiling_low_peak.dropna(), bins=20, color="C0")
ax[1].set(xlabel="peak split-half r", ylabel="people", title="Noise ceiling at each person's peak")
fig.tight_layout()
fig.savefig(C.FIGURES_DIR / "08a_noise_ceiling.png", dpi=150)

## Self-check

The first five rows must hold. The last two are predictions.

In [ ]:
conv = audit[~audit.get("loaded from an earlier run", pd.Series(False, index=audit.index)).fillna(False).astype(bool)]
rows = [
    {"measure": "people converted", "measured": float(len(audit) - audit.get("skipped", pd.Series(dtype=object)).notna().sum()), "expected": "49 (sub-06 skipped: incomplete training set)", "ok": (49, 49), "look_again": "the time budget cut the run; rerun with this output attached"},
    {"measure": "people with validation data", "measured": float(audit.has_validation.sum()), "expected": "48 (all but 1 and 6)", "ok": (48, 48), "look_again": "a validation part is missing"},
    {"measure": "people whose training images equal the shared list", "measured": float(sum(same.values())), "expected": "49 (all but sub-06)", "ok": (49, 49), "look_again": "another person saw other images; read their events table"},
    {"measure": "fewest repetitions of a validation image", "measured": float(conv.test_reps_min.min()) if "test_reps_min" in conv else np.nan, "expected": "12", "ok": (12, 12), "look_again": "some repetitions are missing"},
    {"measure": "stimulus events in the EEG minus rows in the table, largest", "measured": float((conv.events_eeg - conv.events_table).max()) if len(conv) else np.nan, "expected": "0", "ok": (0, 0),
     "look_again": "extra 'E  1' events: the first n are paired with the table as the authors do, but check the pairing"},
    {"measure": "noise ceiling at the peak, median over people", "measured": float(audit.noise_ceiling_low_peak.median()), "expected": "0.08 to 0.30", "ok": (0.08, 0.30),
     "look_again": "the validation responses are more or less reliable than THINGS-EEG2's 80 repetitions predict for 12 (0.22 by Spearman-Brown)"},
    {"measure": "time of the mean ceiling's peak, ms", "measured": float(times_ms[int(nc.mean(0).argmax())]), "expected": "80 to 200", "ok": (80, 200),
     "look_again": "the epochs are shifted: check the event pairing and the downsampling"},
]
status = check("08a", rows, C.METRICS_DIR)
headline("08a", {"people": len(audit), "with validation": int(audit.has_validation.sum()), "ceiling median": float(audit.noise_ceiling_low_peak.median()),
                 "ceiling peak ms": int(times_ms[int(nc.mean(0).argmax())]), "hours": round((time.time() - T0) / 3600, 2)}, C.METRICS_DIR)

## Save outputs as a Kaggle Dataset

Publishes every file this notebook wrote to `data/processed/`, `results/metrics/` and `results/figures/` as a new version of the private Kaggle Dataset `cross-subject-brain-encoding-08a-outputs` (created on the first run). The cell first checks that the required outputs exist, so an unfinished run uploads nothing.

Needs Internet on and the Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` attached to this notebook (Add-ons -> Secrets). Without them the cell prints how to set them up and skips.

In [ ]:
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

EXPORT_TO_DATASET = True
if EXPORT_TO_DATASET:
    export_to_dataset(
        slug="cross-subject-brain-encoding-08a-outputs",
        title="cross-subject-brain-encoding 08a outputs",
        files=[C.DATA_PROCESSED / "eeg1_image_metadata.npy", C.DATA_PROCESSED / "eeg1_k_subsets.json", C.METRICS_DIR / "08a_audit.csv", C.METRICS_DIR / "08a_noise_ceiling.npz", C.METRICS_DIR / "08a_selfcheck.csv", C.METRICS_DIR / "08a_headline.json", C.FIGURES_DIR / "08a_noise_ceiling.png"],
        include_dirs=[C.DATA_PROCESSED, C.METRICS_DIR, C.FIGURES_DIR],  # everything else this notebook wrote
        message="notebook run",
    )